## Purpose:
Load the raw KoboToolbox export, rename columns to clean snake_case, drop Kobo metadata clutter, resolve the GPS inconsistency across facilities, and encode the structural (by-design) missingness in delay_reason/days_late explicitly rather than treating it as a gap.

In [1]:
import pandas as pd
import numpy as np
from config import RAW_PATH, RENAME_MAP, KEEP_COLUMNS, DISCREPANCY_FLAG_COLUMNS, CLEAN_PATH


In [2]:
# 1. Load
df = pd.read_excel(RAW_PATH)
print(f"Raw shape: {df.shape}")


Raw shape: (41, 33)


In [3]:
df.head(5)

,Sub-county,Facility name,Date of visit,Supervisor name,Was last month's report submitted on time?,"If no, reason for delay",Days late (if applicable),Were registers complete for the reporting period?,Number of discrepancies found (register vs report),Type of discrepancy (select all that apply),...,_uuid,_submission_time,_validation_status,_notes,_status,_submitted_by,__version__,_tags,meta/rootUuid,_index
0,Ainabkoi,Burnt Forest Hospital,2026-08-04,Joy Alisa,Yes,NaN,NaN,"Yes, complete",4,Miscounted totals,...,8c8db3cd-29b3-4fea-8900-a5a0c26c322a,2026-08-12T13:49:14,NaN,NaN,submitted_via_web,jentrix123,vodsrLh3q998LETSSyJ4ym,NaN,uuid:8c8db3cd-29b3-4fea-8900-a5a0c26c322a,1
1,Turbo,Huruma Sub-County Hospital,2026-08-06,Anna Cheptoo,Yes,NaN,NaN,Partially complete,14,Wrong indicator used,...,b16386f6-7088-4227-a9f6-49acea620349,2026-08-12T13:56:16,NaN,NaN,submitted_via_web,jentrix123,vodsrLh3q998LETSSyJ4ym,NaN,uuid:c49f7c7c-115a-4159-a4fb-97cdf56a656b,2
2,Kapseret,Kapteldon Sub-County Hospital,2026-07-15,Asha Alima,No,Staff shortage,31.0,No,5,Missing entries Miscounted totals,...,409ee14f-f1ab-46dd-8ed0-d8964d7a9dfa,2026-08-12T14:39:01,NaN,NaN,submitted_via_web,jentrix123,vodsrLh3q998LETSSyJ4ym,NaN,uuid:409ee14f-f1ab-46dd-8ed0-d8964d7a9dfa,3
3,Kapseret,Langas Sub-County Hospital,2026-08-12,Angela Bale,Yes,NaN,NaN,Partially complete,11,NaN,...,c4c15260-4190-4d70-9211-e3b95eb8cacf,2026-08-12T15:03:08,NaN,NaN,submitted_via_web,NaN,vodsrLh3q998LETSSyJ4ym,NaN,uuid:c4c15260-4190-4d70-9211-e3b95eb8cacf,4
4,Ainabkoi,Calcare Hospital,2026-06-17,Angela Bale,No,Facility overwhelmed by patient load,8.0,"Yes, complete",3,Miscounted totals Illegible records Wrong indi...,...,7ac08b76-d918-486e-a050-862da03589c7,2026-08-12T17:53:20,NaN,NaN,submitted_via_web,NaN,vodsrLh3q998LETSSyJ4ym,NaN,uuid:7ac08b76-d918-486e-a050-862da03589c7,5


In [4]:
# 2. Rename + keep only needed columns
df = df.rename(columns=RENAME_MAP)[KEEP_COLUMNS].copy()


In [5]:
# 3. Clean discrepancy-type boolean flags (1.0/0.0/NaN -> clean int)
for c in DISCREPANCY_FLAG_COLUMNS:
    df[c] = df[c].fillna(0).astype(int)

In [6]:
# 4. Encode structural missingness explicitly
on_time_mask = df["report_submitted"].str.strip().str.lower() == "yes"
df.loc[on_time_mask, "delay_reason"] = "N/A - submitted on time"
df.loc[on_time_mask, "days_late"] = 0
df["days_late"] = df["days_late"].astype(int)

remaining_missing = df.loc[~on_time_mask, ["delay_reason", "days_late"]].isna().sum()
print("Genuine remaining gaps among late submissions:\n", remaining_missing)


Genuine remaining gaps among late submissions:
 delay_reason    1
days_late       0
dtype: int64


## Missingness validation result

Out of the 16 submissions marked as "not submitted on time," **1 row** is
missing its `delay_reason`, and **0 rows** are missing `days_late`.

This confirms the structural-missingness fill worked correctly: `delay_reason`
and `days_late` are only genuinely missing when they should have been filled
in but weren't — not as a side effect of the earlier fill logic. The single
missing `delay_reason` is a real, standalone data quality gap (a supervisor
selected "No" but didn't record why) rather than an error introduced during
cleaning.

**Decision:** this gap is left as `NaN` rather than filled in, since inventing
a reason would misrepresent what was actually collected. It's noted here as a
legitimate limitation of the dataset — one missing explanation out of 16 late
submissions.

In [7]:
# 5. Resolve GPS inconsistency -> one canonical location per facility
canonical = (
    df.groupby("facility_name")[["gps_lat", "gps_lon"]]
    .apply(lambda g: pd.Series({
        "gps_lat_canonical": g["gps_lat"].mode().iloc[0] if not g["gps_lat"].mode().empty else np.nan,
        "gps_lon_canonical": g["gps_lon"].mode().iloc[0] if not g["gps_lon"].mode().empty else np.nan,
    }))
    .reset_index()
)
df = df.merge(canonical, on="facility_name", how="left")
df["gps_lat"], df["gps_lon"] = df["gps_lat_canonical"], df["gps_lon_canonical"]
df = df.drop(columns=["gps_lat_canonical", "gps_lon_canonical"])

In [8]:
# Display facility name along with updated GPS coordinates
df[["facility_name", "gps_lat", "gps_lon"]].head(10)

,facility_name,gps_lat,gps_lon
0,Burnt Forest Hospital,0.351000,35.368000
1,Huruma Sub-County Hospital,0.638576,35.117072
2,Kapteldon Sub-County Hospital,0.479000,35.245000
3,Langas Sub-County Hospital,0.487000,35.261000
4,Calcare Hospital,0.363000,35.354000
5,Moi's Bridge Mission Hospital,0.719000,35.197000
6,Moi Teaching & Referral Hospital,0.514300,35.269800
7,Burnt Forest Hospital,0.351000,35.368000
8,Kesses Sub-County Hospital,0.418000,35.339000
9,Kapteldon Sub-County Hospital,0.479000,35.245000


In [9]:
# Count unique GPS coordinates per facility (should all be 1)
df.groupby("facility_name")[["gps_lat", "gps_lon"]].nunique()

,gps_lat,gps_lon
facility_name,,
Burnt Forest Hospital,1,1
Calcare Hospital,1,1
Huruma Sub-County Hospital,1,1
Kamara Hospital,1,1
Kapteldon Sub-County Hospital,1,1
Kesses Sub-County Hospital,1,1
Langas Sub-County Hospital,1,1
Moi Teaching & Referral Hospital,1,1
Moi's Bridge Mission Hospital,1,1


### Each facility has a unique 1 pair of  GPS cordinates, no duplicates or incnsistencies left

In [10]:
# Check how many missing coordinates remain
df[["facility_name", "gps_lat", "gps_lon"]].isna().sum()

facility_name    0
gps_lat          0
gps_lon          0
dtype: int64

In [11]:
# 6. Range validation (should already pass -- form enforced these at entry)
assert df["quality_score"].between(1, 5).all()
assert df["discrepancy_count"].between(0, 50).all()
assert df["days_late"].between(0, 31).all()

print("All range validation checks passed successfully!")

All range validation checks passed successfully!


In [12]:
# Inspect min, max, and percentiles for validation columns
df[["quality_score", "discrepancy_count", "days_late"]].describe().T[["min", "mean", "max"]]

,min,mean,max
quality_score,1.0,3.268293,5.0
discrepancy_count,0.0,6.878049,43.0
days_late,0.0,2.829268,31.0


In [13]:
# 7. Dtypes + save
df["visit_date"] = pd.to_datetime(df["visit_date"])
df["submission_time"] = pd.to_datetime(df["submission_time"])
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH} | shape: {df.shape}")

Saved: .\facility_dq_clean.csv | shape: (41, 20)
